# 06.4 Assignment and Augmented Assignment

`x += 1` looks like pure shorthand for `x = x + 1`. For immutable types it is.
For mutable types it is a **different operation** — and that difference causes
real bugs.

**35 numbered examples.**

## Theory

### The augmented operators

Every arithmetic and bitwise operator has an augmented form:

```
+=   -=   *=   /=   //=   %=   **=
&=   |=   ^=   <<=  >>=   @=
```

### What `+=` actually does

Python tries two things, in order:

1. Call `__iadd__` (in-place add) if the type defines it
2. Otherwise fall back to `__add__` and **rebind** the name

Mutable types define `__iadd__`; immutable types cannot. So:

```python
items += [4]        # mutates the list - aliases see it
items = items + [4] # builds a new list - aliases do not
```

Same visible result on the variable, completely different effect on anything
else pointing at that object.

### Why it matters

This is the 04.2 aliasing problem in operator form. If you pass a list into a
function and use `+=`, you have modified the caller's data. If you use `= +`,
you have not.

### The other assignment forms

- **Chained:** `a = b = 0` — one object, several names
- **Unpacking:** `a, b = 1, 2`
- **Starred:** `first, *rest = items`
- **Annotated:** `count: int = 0`
- **Walrus:** `if (n := len(x)) > 3:` — covered fully in 06.9

In [ ]:
# EXAMPLE 1-12: every augmented operator.
print("EXAMPLE 1-12: the augmented operators")
print("")

value = 10
print("   starting value:", value)
print("")

operations = [
    ("1.  +=  5", lambda v: v + 5),
    ("2.  -=  3", lambda v: v - 3),
    ("3.  *=  2", lambda v: v * 2),
    ("4.  /=  4", lambda v: v / 4),
    ("5.  //= 3", lambda v: v // 3),
    ("6.  %=  4", lambda v: v % 4),
    ("7.  **= 2", lambda v: v ** 2),
]

for label, operation in operations:
    current = 10
    current = operation(current)
    print(f"   {label}  ->  {current}")

print("")
print("   Bitwise forms (covered in 06.5):")
bitwise = [
    ("8.  &=  6", 10 & 6),
    ("9.  |=  6", 10 | 6),
    ("10. ^=  6", 10 ^ 6),
    ("11. <<= 2", 10 << 2),
    ("12. >>= 2", 10 >> 2),
]
for label, result in bitwise:
    print(f"   {label}  ->  {result}")

In [ ]:
# EXAMPLE 13-18: += on immutable types REBINDS.
print("EXAMPLE 13-18: immutable types - += rebinds")
print("")

# 13. Integers.
number = 10
id_before = id(number)
number += 5
print("   13. int:")
print("       value:", number, " same object?", id(number) == id_before)

# 14. Strings.
text = "hello"
id_before = id(text)
text += " world"
print("")
print("   14. str:")
print("       value:", repr(text), " same object?", id(text) == id_before)

# 15. Tuples.
pair = (1, 2)
id_before = id(pair)
pair += (3,)
print("")
print("   15. tuple:")
print("       value:", pair, " same object?", id(pair) == id_before)

# 16-18. An alias never sees the change.
original = "start"
alias = original
original += " changed"

print("")
print("   16. alias behaviour with immutables:")
print("       original:", repr(original))
print("       alias:   ", repr(alias), "<- unchanged")
print("")
print("   17. Because += had to build a new object.")
print("   18. For immutable types, += IS just shorthand.")

In [ ]:
# EXAMPLE 19-24: += on mutable types MUTATES.
print("EXAMPLE 19-24: mutable types - += mutates in place")
print("")

# 19. A list keeps its identity.
items = [1, 2]
id_before = id(items)
items += [3]
print("   19. list with +=:")
print("       value:", items, " same object?", id(items) == id_before)

# 20. But `= +` does not.
items = [1, 2]
id_before = id(items)
items = items + [3]
print("")
print("   20. list with = +:")
print("       value:", items, " same object?", id(items) == id_before)

# 21-22. The alias difference.
original = [1, 2]
alias = original
original += [3]
print("")
print("   21. += with an alias:")
print("       original:", original)
print("       alias:   ", alias, "<- ALSO changed")

original = [1, 2]
alias = original
original = original + [3]
print("")
print("   22. = + with an alias:")
print("       original:", original)
print("       alias:   ", alias, "<- unchanged")

# 23-24. Which types have __iadd__.
print("")
print("   23. which types define __iadd__:")
for label, sample in [("list", []), ("str", ""), ("int", 0),
                      ("tuple", ()), ("bytearray", bytearray())]:
    print(f"       {label:<10} {hasattr(type(sample), '__iadd__')}")

print("")
print("   24. That single method decides mutate-vs-rebind.")

In [ ]:
# EXAMPLE 25-28: the function-argument consequence.
print("EXAMPLE 25-28: += inside a function")
print("")


def append_with_iadd(items):
    """Use += - modifies the caller's list."""
    items += ["added"]


def append_with_plus(items):
    """Use = + - leaves the caller's list alone."""
    items = items + ["added"]


# 25. += changes the caller's data.
caller_list = ["original"]
append_with_iadd(caller_list)
print("   25. after append_with_iadd:", caller_list, "<- changed")

# 26. = + does not.
caller_list = ["original"]
append_with_plus(caller_list)
print("   26. after append_with_plus:", caller_list, "<- unchanged")

# 27. The same trap with a tuple containing a list.
container = (1, [2, 3])
print("")
print("   27. tuple containing a list:")
try:
    container[1] += [4]
except TypeError as error:
    print("       TypeError:", error)
print("       but the list WAS modified:", container)
print("       += mutated first, THEN failed to reassign the tuple slot")

# 28. The lesson.
print("")
print("   28. If a function must not change its argument, do not use +=")
print("       on a mutable parameter. Build a new object instead.")

## The other assignment forms

In [ ]:
# EXAMPLE 29-35: assignment forms.
print("EXAMPLE 29-35: assignment forms")
print("")

# 29. Chained - one object, several names.
a = b = c = []
a.append("shared")
print("   29. a = b = c = []")
print("       after a.append:", c, " same object?", a is b is c)

# 30. Independent assignment.
d, e = [], []
d.append("only d")
print("")
print("   30. d, e = [], []")
print("       d:", d, " e:", e, " same object?", d is e)

# 31. Tuple unpacking.
x, y = 1, 2
print("")
print("   31. x, y = 1, 2 ->", x, y)

# 32. Swap.
x, y = y, x
print("   32. after x, y = y, x ->", x, y)

# 33. Starred unpacking.
first, *middle, last = [1, 2, 3, 4, 5]
print("   33. first, *middle, last ->", first, middle, last)

# 34. Annotated assignment.
count: int = 0
print("   34. count: int = 0 ->", count)
print("       annotations are not enforced at runtime:")
count = "now a string"
print("       count =", repr(count), "- no error")

# 35. Multiple targets in one statement.
values = [0, 0, 0]
index = 0
index, values[index] = 2, 99
print("")
print("   35. index, values[index] = 2, 99")
print("       index:", index, " values:", values)
print("       99 landed at position 0 - the OLD index was used,")
print("       because the right side is evaluated before any assignment.")

## Takeaways

1. Every arithmetic and bitwise operator has an **augmented form**.
2. `+=` calls `__iadd__` when the type defines it, otherwise falls back to
   `__add__` and **rebinds**.
3. **Mutable types mutate in place**; immutable types build a new object. Same
   syntax, different effect.
4. That means `items += [x]` inside a function **modifies the caller's list**,
   while `items = items + [x]` does not.
5. `tuple[1] += [4]` both **mutates the list and raises `TypeError`** — the
   mutation happens before the failed reassignment.
6. `a = b = []` binds **one object** to all three names.
7. In multiple assignment, the **right side is fully evaluated first**.
8. Annotations like `count: int = 0` are **not enforced** at runtime.

## Try it yourself

1. Run `id()` before and after `+=` on an int, a str, a list and a tuple.
2. Write a function using `+=` on a list parameter. Prove the caller sees it.
3. Try `t = (1, [2]); t[1] += [3]`. Explain both the error and the mutation.
4. Predict `a = b = []; a.append(1); print(b)`.
5. Find which types have `__iadd__` using `hasattr`.